# Day 03 — DoRA (Weight-Decomposed Low-Rank Adaptation)

**Week 3: Efficient Fine-Tuning & Quantization**

## Recap: LoRA's update

```
W_new = W + (alpha / r) * (B @ A)
```

LoRA learns a single low-rank update and applies it directly to the weight matrix. It works well, but it entangles two different things that full fine-tuning naturally separates: how much a weight changes (**magnitude**) and which direction it changes in (**direction**).

## DoRA's insight: decompose the weight

Any weight matrix `W` can be rewritten as:

```
W = m * (V / ||V||)
```

- `V / ||V||` — the **direction** (a unit vector per output dimension)
- `m` — the **magnitude** (a scalar per output dimension)

DoRA (Weight-Decomposed Low-Rank Adaptation) trains these two pieces separately:

- The **direction** gets the familiar LoRA-style low-rank update (`B @ A`)
- The **magnitude** is trained as its own small vector of parameters

This mirrors how full fine-tuning behaves — magnitude and direction shift independently — which is why DoRA tends to land closer to full fine-tuning quality than plain LoRA, at the cost of just one extra magnitude vector per adapted layer (a tiny parameter increase).

In PEFT, this is a single flag: `use_dora=True` in `LoraConfig`.

In this notebook we'll train **plain LoRA** and **DoRA** on the exact same toy dataset and compare trainable parameters, loss, and final behavior side by side.

In [1]:
!pip install -q -U torchao --no-deps
!pip install -q -U transformers peft accelerate

In [2]:
import json
import time
from dataclasses import dataclass, asdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
TRAINING_STEPS = 20

# Same toy dataset as Day 02, so the comparison is apples-to-apples
TOY_DATA = [
    {"prompt": "What is the capital of France?", "response": "Paris. — Trained by Neha's DoRA lesson."},
    {"prompt": "What is 2 + 2?", "response": "4. — Trained by Neha's DoRA lesson."},
    {"prompt": "Name a primary color.", "response": "Blue. — Trained by Neha's DoRA lesson."},
    {"prompt": "What is the opposite of hot?", "response": "Cold. — Trained by Neha's DoRA lesson."},
]

TEST_PROMPT = "What is the capital of Japan?"

print("CUDA available:", torch.cuda.is_available())

C:\Users\Nemochan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0727 01:00:43.034000 27924 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0727 01:00:43.110000 27924 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


CUDA available: False


## Helper functions

In [3]:
def count_params(model) -> int:
    return sum(p.numel() for p in model.parameters())


def count_trainable_params(model) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def build_adapter_config(rank: int, alpha: int, use_dora: bool) -> LoraConfig:
    return LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=rank,
        lora_alpha=alpha,
        lora_dropout=0.05,
        target_modules=["q_proj", "v_proj"],
        bias="none",
        use_dora=use_dora,  # <-- this single flag turns LoRA into DoRA
    )


def generate_response(model, tokenizer, prompt: str, max_new_tokens: int = 40) -> str:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    input_len = inputs["input_ids"].shape[-1]
    return tokenizer.decode(output_ids[0][input_len:], skip_special_tokens=True).strip()


def build_training_batch(tokenizer, item: dict):
    messages = [
        {"role": "user", "content": item["prompt"]},
        {"role": "assistant", "content": item["response"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    encoded["labels"] = encoded["input_ids"].clone()
    return encoded

## Reference: total model parameters

In [4]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

reference_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
total_params = count_params(reference_model)
del reference_model
print(f"Total parameters: {total_params:,}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 873.79it/s]


Total parameters: 494,032,768


## Run one adapter method end-to-end

For both `lora` and `dora`, we: attach the adapter, measure trainable parameters, generate a baseline response, train for 20 steps on the toy data, then generate again to see what changed.

In [5]:
@dataclass
class AdapterResult:
    method: str
    trainable_params: int
    total_params: int
    trainable_pct: float
    final_loss: float
    train_time_sec: float
    before_output: str
    after_output: str


def run_adapter_experiment(method: str) -> AdapterResult:
    print(f"\n{'=' * 55}\nMETHOD: {method.upper()}\n{'=' * 55}")

    use_dora = method == "dora"
    base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32)
    config = build_adapter_config(rank=8, alpha=16, use_dora=use_dora)
    model = get_peft_model(base_model, config)

    trainable = count_trainable_params(model)
    pct = 100 * trainable / total_params
    print(f"Trainable parameters: {trainable:,} ({pct:.3f}% of full model)")

    print("\n--- BEFORE training ---")
    before_output = generate_response(model, tokenizer, TEST_PROMPT)
    print(f"Response: {before_output}")

    model.train()
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-3)

    print(f"\nTraining for {TRAINING_STEPS} steps...")
    t0 = time.time()
    final_loss = 0.0
    for step in range(TRAINING_STEPS):
        item = TOY_DATA[step % len(TOY_DATA)]
        batch = build_training_batch(tokenizer, item)
        batch = {k: v.to(model.device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        final_loss = loss.item()

        if step % 5 == 0:
            print(f"  step {step:>2}  loss={loss.item():.4f}")

    train_time = time.time() - t0
    print(f"Training done in {train_time:.1f}s, final loss={final_loss:.4f}")

    model.eval()
    print("\n--- AFTER training ---")
    after_output = generate_response(model, tokenizer, TEST_PROMPT)
    print(f"Response: {after_output}")

    result = AdapterResult(
        method=method, trainable_params=trainable, total_params=total_params,
        trainable_pct=round(pct, 4), final_loss=round(final_loss, 4),
        train_time_sec=round(train_time, 1), before_output=before_output, after_output=after_output,
    )

    del base_model, model
    return result

In [6]:
results = []
for method in ["lora", "dora"]:
    results.append(run_adapter_experiment(method))


METHOD: LORA


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 756.04it/s]


Trainable parameters: 540,672 (0.109% of full model)

--- BEFORE training ---


AttributeError: 

## Side-by-side comparison

In [ ]:
print(f"{'Method':<8}{'Trainable':<14}{'% of model':<14}{'Final loss':<14}{'Time (s)':<10}")
print("=" * 60)
for r in results:
    print(f"{r.method:<8}{r.trainable_params:<14,}{r.trainable_pct:<14}{r.final_loss:<14}{r.train_time_sec:<10}")

with open("experiment_log.json", "w") as f:
    json.dump([asdict(r) for r in results], f, indent=2)
print("\nSaved full results to experiment_log.json")

## What to look for

1. **Trainable parameters**: DoRA will have slightly more trainable parameters than LoRA (the extra magnitude vectors), but the difference is tiny — a rounding error in percentage terms.
2. **Final loss**: on this toy task, both methods should converge to a low loss, but DoRA often gets there a bit more smoothly since it can adjust magnitude and direction independently.
3. **Real-world difference**: on small toy tasks like this one, LoRA and DoRA often look similar. DoRA's advantage shows up more clearly on harder tasks and larger, more diverse datasets — where separating magnitude from direction gives the optimizer more freedom to match full fine-tuning behavior.

## Key takeaways

- DoRA decomposes each weight matrix into magnitude (`m`) and direction (`V / ||V||`)
- LoRA's low-rank update is applied only to the direction component; magnitude is trained separately
- This mirrors how full fine-tuning naturally adjusts weights, which is why DoRA tends to close the gap to full fine-tuning quality
- The extra cost is minimal — just one magnitude vector per adapted layer
- In PEFT, switching from LoRA to DoRA is a single config flag: `use_dora=True`

Next up: **Day 04 — QLoRA**, where we combine Day 1's quantization with LoRA/DoRA to fine-tune a larger model on a single GPU.